# PyMC Sampler Comparison: Cython (Metropolis) vs JAX (NUTS)

This notebook compares two PyMC workflows for Bayesian inference in the aDDM using the **same transformed-space posterior** as the reference Bayesian example in `example4_new/addm_inference.ipynb`.

1. **Cython + Metropolis**: PyMC Metropolis targeting the transformed-space posterior with the Cython likelihood
2. **JAX + NUTS**: PyMC + NumPyro NUTS targeting the same transformed-space posterior with the JAX likelihood and gradients


In [ ]:
import os
import sys
import time
import datetime
import numpy as np

# Add src to path
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pymc as pm
import pytensor
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

from pytensor.graph.op import Op

# JAX imports
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap

from efficient_fpt.jax import set_jax_precision

set_jax_precision(True)

print(f"PyMC version: {pm.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

## Configuration

In [ ]:
# =====================================================================
# Configuration
# =====================================================================
DATA_PATH = "../example4_new/addm_data_20251015-163921.npz"

# Data subset (use smaller for testing, larger for real comparison)
NUM_TRIALS = 500  # Number of trials to use

# Sampling configuration
N_DRAWS = 500      # Number of posterior draws per chain
N_TUNE = 200       # Number of tuning steps
N_CHAINS = 2       # Number of chains

# Cython configuration
NUM_THREADS = 32    # OpenMP threads for Cython
PYMC_CORES = 1     # Avoid forking a multithreaded JAX process

# JAX configuration
TRUNC_NUM = 10      # Fixed truncation length used in this notebook
JAX_USE_REMAT = False

# Random seed for reproducibility
RANDOM_SEED = 42

## Load Data

In [ ]:
# =====================================================================
# Load and prepare data using canonical loader
# =====================================================================
from efficient_fpt.io import load_addm_experiment

data = load_addm_experiment(DATA_PATH)
params = data["params"]
decision = data["decision_data"]
covariates = data["covariates"]

# True parameters
TRUE_PARAMS = {
    'eta': float(params["eta"]),
    'kappa': float(params["kappa"]),
    'a': float(params["a"]),
    'b': float(params["b"]),
    'x0': float(params["x0"]),
    'sigma': float(params["sigma"]),
}

# Extract data arrays
r1_full = covariates["r1_data"].astype(np.float64)
r2_full = covariates["r2_data"].astype(np.float64)
flag_full = covariates["flag_data"].astype(np.int32)
sacc_full = covariates["sacc_array_data"].astype(np.float64)
d_full = covariates["d_data"].astype(np.int32)
rt_full = decision["rt_data"].astype(np.float64)
choice_full = decision["choice_data"].astype(np.int32)

num_data_full = len(rt_full)
max_d = int(d_full.max())

# Subset to NUM_TRIALS
idx = slice(0, min(NUM_TRIALS, num_data_full))
r1_data = r1_full[idx]
r2_data = r2_full[idx]
flag_data = flag_full[idx]
sacc_data = sacc_full[idx]
d_data = d_full[idx]
rt_data = rt_full[idx]
choice_data = choice_full[idx]

num_data = len(rt_data)
sigma = TRUE_PARAMS['sigma']
M = float(np.max(rt_data))  # Max RT for constraints

print(f"Data loaded: {num_data} trials, max_d={max_d}")
print(f"Max RT: {M:.4f}")
print(f"\nTrue parameters:")
for k, v in TRUE_PARAMS.items():
    print(f"  {k}: {v}")


## 1. Cython Implementation (Transformed-Space Metropolis)

This uses a custom PyTensor Op wrapping the Cython likelihood, but the PyMC model is written in the **same unconstrained parameterization** as the reference Bayesian notebook. Metropolis therefore samples the transformed variables rather than making random walks directly in constrained parameter space.


In [ ]:
# Import Cython implementation
from efficient_fpt.cython import compute_addm_mean_nll, print_num_threads

print("Cython implementation loaded.")
print_num_threads()

In [ ]:
class LogLikeCython(Op):
    """
    PyTensor Op wrapping the Cython likelihood function.
    
    Input: theta = [eta, kappa, a, b, x0]
    Output: scalar log-likelihood
    
    NOTE: No grad() method defined = gradient-free only!
    """
    itypes = [pt.dvector]
    otypes = [pt.dscalar]

    def __init__(self, rt_data, choice_data, r1_data, r2_data, flag_data,
                 sacc_data, d_data, sigma, n_threads=8):
        self.rt_data = np.asarray(rt_data, dtype=np.float64)
        self.choice_data = np.asarray(choice_data, dtype=np.int32)
        self.r1_data = np.asarray(r1_data, dtype=np.float64)
        self.r2_data = np.asarray(r2_data, dtype=np.float64)
        self.flag_data = np.asarray(flag_data, dtype=np.int32)
        self.sacc_data = np.asarray(sacc_data, dtype=np.float64)
        self.d_data = np.asarray(d_data, dtype=np.int32)
        self.sigma = float(sigma)
        self.num_data = len(self.rt_data)
        self.n_threads = int(n_threads)

    def perform(self, node, inputs, outputs):
        (theta,) = inputs
        eta, kappa, a, b, x0 = theta

        # Compute negative log-likelihood
        nll = compute_addm_mean_nll(
            self.rt_data, self.choice_data,
            eta, kappa, self.sigma, a, b, x0,
            self.r1_data, self.r2_data, self.flag_data,
            self.sacc_data, self.d_data,
            order_mid=30, order_last=30,
            n_threads=self.n_threads,
            log_space=True,
        )
        
        # Return log-likelihood (negative of NLL)
        loglik = -self.num_data * nll
        outputs[0][0] = np.array(loglik, dtype="float64")


# Create the Cython Op
cython_loglike_op = LogLikeCython(
    rt_data=rt_data, choice_data=choice_data,
    r1_data=r1_data, r2_data=r2_data, flag_data=flag_data,
    sacc_data=sacc_data, d_data=d_data,
    sigma=sigma, n_threads=NUM_THREADS,
)

# Test it
theta_true = np.array([TRUE_PARAMS['eta'], TRUE_PARAMS['kappa'], 
                       TRUE_PARAMS['a'], TRUE_PARAMS['b'], TRUE_PARAMS['x0']])
theta_sym = pt.dvector("theta")
ll_cython = pytensor.function([theta_sym], cython_loglike_op(theta_sym))

print(f"Cython log-likelihood at true params: {ll_cython(theta_true):.4f}")

## 2. JAX Implementation (Same Posterior, With Gradients)

This uses the JAX-based likelihood with automatic differentiation, but keeps the **same transformed-space posterior and priors** as the Cython/PyMC model. That makes the sampler comparison much cleaner: same posterior target, different likelihood backend and sampler.


In [ ]:
# Import the public JAX batch NLL factory
from efficient_fpt.jax import make_addm_nll_function_batchscan

print("JAX implementation loaded.")


In [ ]:
# The public batch NLL factory closes over the fixed dataset.
print(
    "Using make_addm_nll_function_batchscan "
    f"(trunc_num={TRUNC_NUM}, use_remat={JAX_USE_REMAT})"
)


In [ ]:
jax_sum_nll = make_addm_nll_function_batchscan(
    rt_data,
    choice_data,
    r1_data,
    r2_data,
    flag_data,
    sacc_data,
    d_data,
    order_mid=30,
    order_last=30,
    trunc_num=TRUNC_NUM,
    log_space=True,
    use_remat=JAX_USE_REMAT,
)

def jax_loglik_batch(eta, kappa, a, b, x0):
    """Compute total log-likelihood for all trials using the public JAX batch API."""
    return -jax_sum_nll(eta, kappa, sigma, a, b, x0)

jax_loglik_jit = jit(jax_loglik_batch)
jax_grad_loglik = jit(grad(jax_loglik_batch, argnums=(0, 1, 2, 3, 4)))

print("Warming up JAX JIT compilation...")
_ = jax_loglik_jit(
    TRUE_PARAMS['eta'],
    TRUE_PARAMS['kappa'],
    TRUE_PARAMS['a'],
    TRUE_PARAMS['b'],
    TRUE_PARAMS['x0'],
)
_ = jax_grad_loglik(
    TRUE_PARAMS['eta'],
    TRUE_PARAMS['kappa'],
    TRUE_PARAMS['a'],
    TRUE_PARAMS['b'],
    TRUE_PARAMS['x0'],
)
print("Done.")

ll_jax = jax_loglik_jit(
    TRUE_PARAMS['eta'],
    TRUE_PARAMS['kappa'],
    TRUE_PARAMS['a'],
    TRUE_PARAMS['b'],
    TRUE_PARAMS['x0'],
)
print(f"JAX log-likelihood at true params: {float(ll_jax):.4f}")


In [ ]:
from pytensor.link.jax.dispatch import jax_funcify

class LogLikeJAX(Op):
    """
    PyTensor Op wrapping the JAX likelihood function WITH GRADIENTS.
    
    This enables NUTS sampling by providing gradient information.
    """
    itypes = [pt.dvector]
    otypes = [pt.dscalar]

    def __init__(self, loglik_fn, grad_fn):
        self.loglik_fn = loglik_fn
        self.grad_fn = grad_fn

    def perform(self, node, inputs, outputs):
        (theta,) = inputs
        eta, kappa, a, b, x0 = theta
        
        # Compute log-likelihood using JAX
        loglik = float(self.loglik_fn(eta, kappa, a, b, x0))
        outputs[0][0] = np.array(loglik, dtype="float64")

    def grad(self, inputs, output_grads):
        """Compute gradient using JAX autodiff."""
        (theta,) = inputs
        (g_out,) = output_grads
        
        # Use custom gradient Op
        grads = LogLikeJAXGrad(self.grad_fn)(theta)
        
        # Chain rule: multiply by output gradient
        return [g_out * grads]


class LogLikeJAXGrad(Op):
    """
    Gradient Op for the JAX likelihood.
    """
    itypes = [pt.dvector]
    otypes = [pt.dvector]

    def __init__(self, grad_fn):
        self.grad_fn = grad_fn

    def perform(self, node, inputs, outputs):
        (theta,) = inputs
        eta, kappa, a, b, x0 = theta
        
        # Compute gradients using JAX
        grads = self.grad_fn(eta, kappa, a, b, x0)
        grad_array = np.array([float(g) for g in grads], dtype="float64")
        outputs[0][0] = grad_array


# Register JAX implementations for our custom Ops
# This allows numpyro/blackjax backends to use them
@jax_funcify.register(LogLikeJAX)
def jax_funcify_LogLikeJAX(op, **kwargs):
    """Convert LogLikeJAX Op to a pure JAX function."""
    loglik_fn = op.loglik_fn
    
    def loglik_jax(theta):
        eta, kappa, a, b, x0 = theta[0], theta[1], theta[2], theta[3], theta[4]
        return loglik_fn(eta, kappa, a, b, x0)
    
    return loglik_jax


@jax_funcify.register(LogLikeJAXGrad)
def jax_funcify_LogLikeJAXGrad(op, **kwargs):
    """Convert LogLikeJAXGrad Op to a pure JAX function."""
    grad_fn = op.grad_fn
    
    def grad_jax(theta):
        eta, kappa, a, b, x0 = theta[0], theta[1], theta[2], theta[3], theta[4]
        grads = grad_fn(eta, kappa, a, b, x0)
        return jnp.stack(grads)
    
    return grad_jax


# Create the JAX Op with gradients
jax_loglike_op = LogLikeJAX(jax_loglik_jit, jax_grad_loglik)

# Test
ll_jax_op = pytensor.function([theta_sym], jax_loglike_op(theta_sym))
print(f"JAX Op log-likelihood at true params: {ll_jax_op(theta_true):.4f}")

# Test gradient
grad_sym = pt.grad(jax_loglike_op(theta_sym), theta_sym)
grad_fn = pytensor.function([theta_sym], grad_sym)
print(f"JAX Op gradients at true params: {grad_fn(theta_true)}")

## 3. Compare Likelihood Evaluation Speed

In [ ]:
def time_fn(fn, *args, n=20):
    """Time a function over n calls."""
    start = time.perf_counter()
    for _ in range(n):
        _ = fn(*args)
    elapsed = time.perf_counter() - start
    return elapsed / n

print("=" * 60)
print("Likelihood Evaluation Timing")
print("=" * 60)

# Cython timing
t_cython = time_fn(ll_cython, theta_true, n=20)
print(f"Cython:        {t_cython*1000:.2f} ms per evaluation")

# JAX timing (likelihood only)
t_jax = time_fn(lambda t: float(jax_loglik_jit(t[0], t[1], t[2], t[3], t[4])), theta_true, n=20)
print(f"JAX (loglik):  {t_jax*1000:.2f} ms per evaluation ({t_cython/t_jax:.1f}x vs Cython)")

# JAX timing (likelihood + gradient)
def jax_loglik_and_grad(theta):
    ll = jax_loglik_jit(theta[0], theta[1], theta[2], theta[3], theta[4])
    g = jax_grad_loglik(theta[0], theta[1], theta[2], theta[3], theta[4])
    return ll, g

t_jax_grad = time_fn(jax_loglik_and_grad, theta_true, n=20)
print(f"JAX (+grad):   {t_jax_grad*1000:.2f} ms per evaluation")

## 4. Build PyMC Models

In [ ]:
def z_to_u_np(z):
    """Map constrained parameters z=(eta, kappa, a, b, x0) to unconstrained u-space."""
    eta, kappa, a, b, x0 = [float(v) for v in z]
    tb = b * M / a
    sx = (x0 + a) / (2.0 * a)
    return np.array([
        np.log(eta) - np.log1p(-eta),
        np.log(kappa),
        np.log(a),
        np.log(tb) - np.log1p(-tb),
        np.log(sx) - np.log1p(-sx),
    ], dtype=np.float64)


def build_pymc_model(loglike_op, name_suffix=""):
    """
    Build a PyMC model on the same unconstrained parameterization used by
    the reference Bayesian notebook in example4_new/addm_inference.ipynb.
    """
    init_u = z_to_u_np([
        TRUE_PARAMS["eta"], TRUE_PARAMS["kappa"], TRUE_PARAMS["a"],
        TRUE_PARAMS["b"], TRUE_PARAMS["x0"],
    ])

    with pm.Model() as model:
        # Unconstrained variables sampled by PyMC
        u_eta = pm.Flat(f"u_eta{name_suffix}", initval=float(init_u[0]))
        u_kappa = pm.Flat(f"u_kappa{name_suffix}", initval=float(init_u[1]))
        u_a = pm.Flat(f"u_a{name_suffix}", initval=float(init_u[2]))
        u_b = pm.Flat(f"u_b{name_suffix}", initval=float(init_u[3]))
        u_x0 = pm.Flat(f"u_x0{name_suffix}", initval=float(init_u[4]))

        # Constrained parameters
        eta = pm.Deterministic(f"eta{name_suffix}", pm.math.sigmoid(u_eta))
        kappa = pm.Deterministic(f"kappa{name_suffix}", pt.exp(u_kappa))
        a = pm.Deterministic(f"a{name_suffix}", pt.exp(u_a))
        tb = pm.math.sigmoid(u_b)
        b = pm.Deterministic(f"b{name_suffix}", (a / M) * tb)
        sx = pm.math.sigmoid(u_x0)
        x0 = pm.Deterministic(f"x0{name_suffix}", -a + 2.0 * a * sx)

        theta = pt.stack([eta, kappa, a, b, x0])

        # Match the priors from example4_new/addm_inference.ipynb exactly.
        log_prior = (
            pm.logp(pm.Beta.dist(alpha=2.0, beta=2.0), eta)
            + pm.logp(pm.Gamma.dist(alpha=2.0, beta=4.0), kappa)
            + pm.logp(pm.Gamma.dist(alpha=2.0, beta=1.0), a)
            + pm.logp(pm.Beta.dist(alpha=2.0, beta=2.0), tb) - pt.log(a / M)
            + pm.logp(pm.Beta.dist(alpha=2.0, beta=2.0), sx) - pt.log(2.0 * a)
        )

        # Jacobian for z(u), matching the transformed-space MH reference.
        log_jac = (
            pt.log(eta) + pt.log1p(-eta)
            + pt.log(kappa)
            + pt.log(a)
            + pt.log(a / M) + pt.log(tb) + pt.log1p(-tb)
            + pt.log(2.0 * a) + pt.log(sx) + pt.log1p(-sx)
        )

        pm.Potential(f"logpost{name_suffix}", loglike_op(theta) + log_prior + log_jac)

    return model


print("Building PyMC models on transformed space...")
model_cython = build_pymc_model(cython_loglike_op, "_cy")
model_jax = build_pymc_model(jax_loglike_op, "_jax")
print("Done.")


## 5. Run Cython + Metropolis Sampling on the Transformed Posterior


In [ ]:
print("=" * 60)
print("Cython + Metropolis Sampling")
print("=" * 60)
print(f"Configuration: {N_DRAWS} draws, {N_TUNE} tune, {N_CHAINS} chains")
print(f"Started at: {datetime.datetime.now().strftime('%H:%M:%S')}")

with model_cython:
    t_start = time.perf_counter()

    trace_cython = pm.sample(
        draws=N_DRAWS,
        tune=N_TUNE,
        chains=N_CHAINS,
        cores=PYMC_CORES,
        # PyMC builds one scalar Metropolis step per unconstrained variable here,
        # so the transformed-space random-walk scale must be a scalar, not a length-5 vector.
        step=pm.Metropolis(
            vars=[
                model_cython['u_eta_cy'], model_cython['u_kappa_cy'],
                model_cython['u_a_cy'], model_cython['u_b_cy'], model_cython['u_x0_cy'],
            ],
            scaling=0.012,
        ),
        random_seed=RANDOM_SEED,
        progressbar=True,
        return_inferencedata=True,
    )

    t_cython_sample = time.perf_counter() - t_start

print(f"\nFinished at: {datetime.datetime.now().strftime('%H:%M:%S')}")
print(f"Total sampling time: {t_cython_sample:.1f} s")
print(f"Time per sample: {t_cython_sample / (N_DRAWS * N_CHAINS) * 1000:.1f} ms")


## 6. Run JAX + NUTS Sampling on the Same Transformed Posterior

Because the JAX likelihood Op provides gradients, we can sample the **same transformed-space posterior** with NUTS.


In [ ]:
print("=" * 60)
print("JAX + NUTS Sampling (numpyro backend)")
print("=" * 60)
print(f"Configuration: {N_DRAWS} draws, {N_TUNE} tune, {N_CHAINS} chains")
print(f"Started at: {datetime.datetime.now().strftime('%H:%M:%S')}")

with model_jax:
    t_start = time.perf_counter()
    
    # Use numpyro backend - our custom Op is registered via jax_funcify
    trace_jax = pm.sample(
        draws=N_DRAWS,
        tune=N_TUNE,
        chains=N_CHAINS,
        nuts_sampler="numpyro",
        nuts_sampler_kwargs={"chain_method": "vectorized"}, # "parallel" is for parallelization across devices
        random_seed=RANDOM_SEED,
        progressbar=True,
        return_inferencedata=True,
    )
    
    t_jax_sample = time.perf_counter() - t_start

print(f"\nFinished at: {datetime.datetime.now().strftime('%H:%M:%S')}")
print(f"Total sampling time: {t_jax_sample:.1f} s")
print(f"Time per sample: {t_jax_sample / (N_DRAWS * N_CHAINS) * 1000:.1f} ms")

## 7. Compare Results

In [ ]:
print("=" * 60)
print("Sampling Performance Comparison")
print("=" * 60)

print(f"\nCython + Metropolis:")
print(f"  Total time: {t_cython_sample:.1f} s")
print(f"  Time per sample: {t_cython_sample / (N_DRAWS * N_CHAINS) * 1000:.1f} ms")

print(f"\nJAX + NUTS:")
print(f"  Total time: {t_jax_sample:.1f} s")
print(f"  Time per sample: {t_jax_sample / (N_DRAWS * N_CHAINS) * 1000:.1f} ms")

print(f"\nSpeedup: {t_cython_sample / t_jax_sample:.2f}x")

In [ ]:
# Rename variables for comparison
trace_cython_renamed = trace_cython.rename({
    'eta_cy': 'eta', 'kappa_cy': 'kappa', 'a_cy': 'a', 'b_cy': 'b', 'x0_cy': 'x0'
})
trace_jax_renamed = trace_jax.rename({
    'eta_jax': 'eta', 'kappa_jax': 'kappa', 'a_jax': 'a', 'b_jax': 'b', 'x0_jax': 'x0'
})

params = ['eta', 'kappa', 'a', 'b', 'x0']

print("\n" + "=" * 60)
print("Parameter Estimates Comparison")
print("=" * 60)

print(f"\n{'Param':<8} {'True':<10} {'Cython Mean':<12} {'JAX Mean':<12} {'Cython ESS':<12} {'JAX ESS':<12}")
print("-" * 68)

for param in params:
    true_val = TRUE_PARAMS[param]
    
    cy_mean = float(trace_cython_renamed.posterior[param].mean())
    jax_mean = float(trace_jax_renamed.posterior[param].mean())
    
    cy_ess = float(az.ess(trace_cython_renamed, var_names=[param])[param])
    jax_ess = float(az.ess(trace_jax_renamed, var_names=[param])[param])
    
    print(f"{param:<8} {true_val:<10.4f} {cy_mean:<12.4f} {jax_mean:<12.4f} {cy_ess:<12.1f} {jax_ess:<12.1f}")

In [ ]:
print("\n" + "=" * 60)
print("Effective Samples per Second")
print("=" * 60)

print(f"\n{'Param':<8} {'Cython ESS/s':<15} {'JAX ESS/s':<15} {'JAX Advantage':<15}")
print("-" * 55)

for param in params:
    cy_ess = float(az.ess(trace_cython_renamed, var_names=[param])[param])
    jax_ess = float(az.ess(trace_jax_renamed, var_names=[param])[param])
    
    cy_ess_per_s = cy_ess / t_cython_sample
    jax_ess_per_s = jax_ess / t_jax_sample
    
    advantage = jax_ess_per_s / cy_ess_per_s
    
    print(f"{param:<8} {cy_ess_per_s:<15.2f} {jax_ess_per_s:<15.2f} {advantage:<15.2f}x")

## 8. Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 16))

for i, param in enumerate(params):
    # Trace plot - Cython
    ax_cy = axes[i, 0]
    for chain in range(N_CHAINS):
        ax_cy.plot(trace_cython_renamed.posterior[param].sel(chain=chain).values, 
                   alpha=0.7, label=f'Chain {chain}')
    ax_cy.axhline(TRUE_PARAMS[param], color='red', linestyle='--', label='True')
    ax_cy.set_ylabel(param)
    ax_cy.set_title(f'Cython + Metropolis: {param}')
    if i == 0:
        ax_cy.legend(loc='upper right')
    
    # Trace plot - JAX
    ax_jax = axes[i, 1]
    for chain in range(N_CHAINS):
        ax_jax.plot(trace_jax_renamed.posterior[param].sel(chain=chain).values,
                    alpha=0.7, label=f'Chain {chain}')
    ax_jax.axhline(TRUE_PARAMS[param], color='red', linestyle='--', label='True')
    ax_jax.set_ylabel(param)
    ax_jax.set_title(f'JAX + NUTS: {param}')
    if i == 0:
        ax_jax.legend(loc='upper right')

axes[-1, 0].set_xlabel('Sample')
axes[-1, 1].set_xlabel('Sample')

plt.tight_layout()
plt.suptitle('Trace Plots: Cython (Metropolis) vs JAX (NUTS)', y=1.02, fontsize=14)
plt.show()

In [ ]:
az.plot_pair(trace_cython_renamed, var_names=params)

In [ ]:
az.plot_pair(trace_jax_renamed, var_names=params)

In [ ]:
# Posterior comparison
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, param in enumerate(params):
    ax = axes[i]
    
    # Cython posterior
    cy_samples = trace_cython_renamed.posterior[param].values.flatten()
    ax.hist(cy_samples, bins=50, alpha=0.5, density=True, label='Cython (Metropolis)')
    
    # JAX posterior
    jax_samples = trace_jax_renamed.posterior[param].values.flatten()
    ax.hist(jax_samples, bins=50, alpha=0.5, density=True, label='JAX (NUTS)')
    
    # True value
    ax.axvline(TRUE_PARAMS[param], color='red', linestyle='--', linewidth=2, label='True')
    
    ax.set_xlabel(param)
    ax.set_ylabel('Density' if i == 0 else '')
    ax.set_title(param)
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.suptitle('Posterior Distributions: Cython vs JAX', y=1.02, fontsize=14)
plt.show()

In [ ]:
# ArviZ summary
print("\n" + "=" * 60)
print("Cython + Metropolis Summary")
print("=" * 60)
print(az.summary(trace_cython_renamed, var_names=params))

In [ ]:
print("\n" + "=" * 60)
print("JAX + NUTS Summary")
print("=" * 60)
print(az.summary(trace_jax_renamed, var_names=params))